# 05 — Final Evaluation on Locked Test Set

**This notebook is run exactly once.**  
The test set has been locked since `01_eda.ipynb`. This is the first and only time it is opened.

Results here are the honest, unbiased performance numbers reported in the model card.

In [ ]:
!pip install -q lightgbm shap

In [ ]:
import pandas as pd
import numpy as np
import pickle
import json
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    confusion_matrix, classification_report,
    precision_recall_curve, roc_curve,
)
import warnings

warnings.filterwarnings('ignore')

print('Loading locked test set for the first time...')
test_raw = pd.read_parquet('/kaggle/working/test_locked.parquet')

In [ ]:
# Apply same feature engineering as notebook 02
import json

with open('/kaggle/working/cols_to_drop.json') as f:
    cols_to_drop = json.load(f)

with open('/kaggle/working/feature_cols.json') as f:
    feature_cols = json.load(f)

with open('/kaggle/working/label_encoders.pkl', 'rb') as f:
    encoders = pickle.load(f)

START_DATE = pd.Timestamp('2017-12-01')

def apply_feature_engineering(df, encoders, cols_to_drop, train_df):
    dt = START_DATE + pd.to_timedelta(df['TransactionDT'], unit='s')
    df['hour_of_day'] = dt.dt.hour
    df['day_of_week'] = dt.dt.dayofweek
    df['is_night'] = ((df['hour_of_day'] < 6) | (df['hour_of_day'] >= 23)).astype(int)
    df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)

    df['TransactionAmt_log'] = np.log1p(df['TransactionAmt'])
    card_mean = train_df.groupby('card1')['TransactionAmt'].mean().rename('card1_amt_mean')
    df = df.merge(card_mean, on='card1', how='left')
    df['amt_vs_card_mean'] = df['TransactionAmt'] / df['card1_amt_mean'].fillna(df['TransactionAmt'])

    df['P_emaildomain'] = df['P_emaildomain'].fillna('unknown')
    df['R_emaildomain'] = df['R_emaildomain'].fillna('unknown')
    HIGH_RISK = {'anonymous.com', 'aim.com', 'protonmail.com'}
    df['p_email_is_high_risk'] = df['P_emaildomain'].isin(HIGH_RISK).astype(int)
    df['r_email_is_high_risk'] = df['R_emaildomain'].isin(HIGH_RISK).astype(int)
    df['email_domain_match'] = (df['P_emaildomain'] == df['R_emaildomain']).astype(int)

    df_sorted = df.sort_values('TransactionDT')
    df['card1_txn_count'] = df_sorted.groupby('card1').cumcount().values
    df_sorted['prev_dt'] = df_sorted.groupby('card1')['TransactionDT'].shift(1)
    df_sorted['time_since_last_txn'] = (df_sorted['TransactionDT'] - df_sorted['prev_dt']) / 3600
    df['time_since_last_txn'] = df_sorted['time_since_last_txn'].values
    df['rapid_repeat'] = (df['time_since_last_txn'] < 0.1).astype(int)

    drop_present = [c for c in cols_to_drop if c in df.columns]
    df = df.drop(columns=drop_present + ['TransactionID', 'TransactionDT'], errors='ignore')

    cat_cols = df.select_dtypes(include='object').columns.tolist()
    for col in cat_cols:
        if col in encoders:
            le = encoders[col]
            df[col] = df[col].astype(str).map(
                lambda x: x if x in le.classes_ else le.classes_[0]
            )
            df[col] = le.transform(df[col])
        else:
            df[col] = 0

    df = df.fillna(-999)
    return df

train_raw = pd.read_parquet('/kaggle/working/train.parquet')
test = apply_feature_engineering(test_raw, encoders, cols_to_drop, train_raw)

available_features = [c for c in feature_cols if c in test.columns]
missing_features   = [c for c in feature_cols if c not in test.columns]
for c in missing_features:
    test[c] = -999

X_test = test[feature_cols].values
y_test = test['isFraud'].values

print(f'Test set shape: {X_test.shape}')
print(f'Test fraud rate: {y_test.mean():.4%}')

In [ ]:
# Load models and generate test predictions
with open('/kaggle/working/lgbm_model.pkl', 'rb') as f:
    lgbm_model = pickle.load(f)

with open('/kaggle/working/meta_learner.pkl', 'rb') as f:
    meta_learner = pickle.load(f)

lgbm_test_preds = lgbm_model.predict(X_test)

mlp_test_preds = lgbm_test_preds  # Fallback if MLP not available
try:
    import torch
    with open('/kaggle/working/mlp_model.pkl', 'rb') as f:
        mlp_model = pickle.load(f)
    with open('/kaggle/working/scaler.pkl', 'rb') as f:
        scaler = pickle.load(f)
    X_test_scaled = scaler.transform(X_test)
    mlp_model.eval()
    with torch.no_grad():
        mlp_test_preds = mlp_model(torch.FloatTensor(X_test_scaled)).numpy()
except Exception as e:
    print(f'MLP not loaded ({e}), using LightGBM only for ensemble')

X_meta_test = np.column_stack([lgbm_test_preds, mlp_test_preds])
final_preds = meta_learner.predict_proba(X_meta_test)[:, 1]

print(f'Test ROC-AUC: {roc_auc_score(y_test, final_preds):.4f}')
print(f'Test PR-AUC:  {average_precision_score(y_test, final_preds):.4f}')

In [ ]:
BLOCK_THRESHOLD = 0.70
REVIEW_THRESHOLD = 0.35

preds_block = (final_preds >= BLOCK_THRESHOLD).astype(int)

print('=== Classification Report at BLOCK threshold (0.70) ===')
print(classification_report(y_test, preds_block, target_names=['Legitimate', 'Fraud']))

cm = confusion_matrix(y_test, preds_block)
tn, fp, fn, tp = cm.ravel()

fpr = fp / (fp + tn)
print(f'False Positive Rate: {fpr:.4%}')
print(f'False Positive count: {fp:,} out of {tn+fp:,} legitimate transactions')
print(f'\nAt 10M daily transactions:')
print(f'  Legitimate transactions: {10_000_000 * (1 - 0.035):,.0f}')
print(f'  Blocked legitimate customers/day: {10_000_000 * (1 - 0.035) * fpr:,.0f}')

In [ ]:
fig = plt.figure(figsize=(16, 10))
gs = gridspec.GridSpec(2, 3)

# PR Curve
ax1 = fig.add_subplot(gs[0, 0])
prec, rec, thresh = precision_recall_curve(y_test, final_preds)
pr_auc = average_precision_score(y_test, final_preds)
ax1.plot(rec, prec, color='crimson', linewidth=2)
ax1.fill_between(rec, prec, alpha=0.1, color='crimson')
ax1.set_title(f'Precision-Recall Curve (PR-AUC={pr_auc:.3f})')
ax1.set_xlabel('Recall')
ax1.set_ylabel('Precision')

# Confusion matrix
ax2 = fig.add_subplot(gs[0, 1])
sns.heatmap(cm, annot=True, fmt=',d', cmap='Blues', ax=ax2,
            xticklabels=['Pred Legit', 'Pred Fraud'],
            yticklabels=['Actual Legit', 'Actual Fraud'])
ax2.set_title(f'Confusion Matrix (threshold={BLOCK_THRESHOLD})')

# Threshold sensitivity — cost
ax3 = fig.add_subplot(gs[0, 2])
thresholds = np.linspace(0.1, 0.95, 85)
costs = []
N_DAILY = 10_000_000
FRAUD_RATE = y_test.mean()
FP_COST = 10
FN_COST = 150

for t in thresholds:
    pb = (final_preds >= t).astype(int)
    _tn, _fp, _fn, _tp = confusion_matrix(y_test, pb).ravel()
    _fpr = _fp / (_fp + _tn + 1e-9)
    _fnr = _fn / (_fn + _tp + 1e-9)
    daily_fp = N_DAILY * (1 - FRAUD_RATE) * _fpr
    daily_fn = N_DAILY * FRAUD_RATE * _fnr
    costs.append(daily_fp * FP_COST + daily_fn * FN_COST)

opt_idx = np.argmin(costs)
ax3.plot(thresholds, [c / 1e6 for c in costs], color='steelblue', linewidth=2)
ax3.axvline(x=thresholds[opt_idx], color='green', linestyle='--',
            label=f'Optimal={thresholds[opt_idx]:.2f}')
ax3.axvline(x=BLOCK_THRESHOLD, color='crimson', linestyle='--',
            label=f'Selected={BLOCK_THRESHOLD}')
ax3.set_title('Daily Business Cost vs Threshold (10M txns/day)')
ax3.set_xlabel('Threshold')
ax3.set_ylabel('Cost ($M/day)')
ax3.legend()

# Score distribution
ax4 = fig.add_subplot(gs[1, :])
ax4.hist(final_preds[y_test == 0], bins=100, alpha=0.6, color='steelblue', label='Legitimate', density=True)
ax4.hist(final_preds[y_test == 1], bins=100, alpha=0.6, color='crimson', label='Fraud', density=True)
ax4.axvline(x=BLOCK_THRESHOLD, color='black', linestyle='--', linewidth=2, label=f'Block: {BLOCK_THRESHOLD}')
ax4.axvline(x=REVIEW_THRESHOLD, color='gray', linestyle='--', linewidth=1.5, label=f'Review: {REVIEW_THRESHOLD}')
ax4.set_title('Score Distribution: Fraud vs Legitimate')
ax4.set_xlabel('Fraud Probability')
ax4.legend()

plt.suptitle('Fraud Detection Model — Final Test Set Evaluation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/kaggle/working/final_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Save metrics for API to serve
prec_at_block = tp / max(tp + fp, 1)
rec_at_block  = tp / max(tp + fn, 1)

metrics = {
    'roc_auc': float(roc_auc_score(y_test, final_preds)),
    'pr_auc': float(average_precision_score(y_test, final_preds)),
    'precision_at_block': float(prec_at_block),
    'recall_at_block': float(rec_at_block),
    'false_positive_rate': float(fpr),
    'block_threshold': BLOCK_THRESHOLD,
    'review_threshold': REVIEW_THRESHOLD,
    'test_fraud_rate': float(y_test.mean()),
    'test_size': int(len(y_test)),
}

with open('/kaggle/working/final_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print('=== FINAL METRICS ===')
for k, v in metrics.items():
    print(f'{k}: {v}')

print('\nUpdate api/main.py MODEL_METRICS dict with these values before deploying.')